# **Part 1: Balanaced SMS-Spam Classification**

---

## **Environment Setup (Local or Google Colab)**

The code block below automatically detects your execution runtime environment (Local vs. Google Colab), clones the repository, and installs all necessary dependencies.

> **Google Colab Note:** Once setup completes, navigate to **Runtime → Restart session** in the top menu before proceeding to the next steps to ensure all newly installed packages are correctly loaded.

In [ ]:
import os
import subprocess
import sys

# Environment Detection: Check sys.modules for Colab runtime
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Google Colab environment. Initializing setup...")

    # Define paths
    branch = "dev"
    repo_url = (
        "https://github.com/paymantohidifar/gpt2-text-classifier-from-scratch.git"
    )
    target_dir = "/content/gpt2-classifier"

    # Clean stale builds and clone main branch
    subprocess.run(f"rm -rf {target_dir}", shell=True, check=True)
    subprocess.run(
        f"git clone {repo_url} --branch {branch} {target_dir}",
        shell=True,
        check=True,
    )
    print("Cloned the repo.")

    # Change working directory
    os.chdir(target_dir)

    # Bootstrap uv and install dependencies into system environment
    install_cmd = (
        'curl -LsSf https://astral.sh/uv/install.sh | sh && '
        'export PATH="$HOME/.local/bin:${PATH}" && '
        'uv pip install -e .[gpu,dev] --system --break-system-packages --color never'
    )
    print("Installing dependencies...")
    subprocess.run(install_cmd, shell=True, check=True)
    print("Colab setup complete.")

else:
    # Local Development: Enable IPython Auto-Reload safely
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
        print("Enabled IPython autoreload.")


# Verify PyTorch version and config
import torch

print(f"\nInstalled Pytorch version: {torch.__version__}")
print(f"PyTorch enabled with Cuda: {torch.cuda.is_available()}")

---

## **Table of Contents**

---

## **Prepare Datasets**

### **Download and Split Datasets**

In [ ]:
from gpt2_classifier.datasets_registry import get_dataset_spec
from gpt2_classifier.data import prepare_dataset

spec = get_dataset_spec("sms-spam")
print(spec)

path = prepare_dataset(spec, balance_labels=True, dataset_split=(0.7, 0.1, 0.2))
print("File path:", path)

### **Create Data Loaders**

In [ ]:
from gpt2_classifier.data import create_data_loaders

train_loader, val_loader, test_loader = create_data_loaders(spec.name)

for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

---

## **Set Up The Model**

## **Initialize GPT-2 Model and Load Weights**

In [ ]:
from gpt2_classifier import paths
from gpt2_classifier.config import get_model_config, URL_DIR
from gpt2_classifier.weights import download_and_load_gpt2, load_weights_into_gpt
from gpt2_classifier.model import GPTModel


model_name = "gpt2-small (124M)"
model_config = get_model_config(model_name)
model_url = f"https://huggingface.co/openai-community/{URL_DIR[model_name]}/resolve/main/model.safetensors"
model_destination = paths.MODELS_DIR / f"{URL_DIR[model_name]}.safetensors"

state_dict = download_and_load_gpt2(model_url, model_destination)
model = GPTModel(model_config)
load_weights_into_gpt(model, state_dict)

print(model)

### **Verify The Loaded Model**

In [ ]:
from gpt2_classifier.utils import generate_response

text_1 = "Every effort moves you"

reponse = generate_response(
    text=text_1,
    model=model,
    max_new_tokens=15
)

print(reponse)

### **Set Trainable Parameters for Finetuning**

In [ ]:
for param in model.parameters():
    param.requires_grad = False

import torch

torch.manual_seed(123)

num_classes = 2

model.out_head = torch.nn.Linear(in_features=model_config['emb_dim'], out_features=num_classes)


for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

print(model)

### **Verify Updated Model's Output**

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(inputs)

device = "cpu"
inputs = inputs.to(device)

with torch.no_grad():
    outputs = model(inputs)
print(outputs)
print(outputs.shape)

---

## **Finetuning The Model**


### **Compute Baseline Metrics: Accuracy**

In [ ]:
from gpt2_classifier.utils import get_device
from gpt2_classifier.evaluate import calc_accuracy_loader

device = get_device()
model.to(device)

torch.manual_seed(123)

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

### **Compute Baseline Metrics: Losses**

In [ ]:
from gpt2_classifier.evaluate import calc_loss_loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

### **Finetuning loop**

In [ ]:
import time
from gpt2_classifier.utils import get_device
from gpt2_classifier.train import get_adam_param_groups
from gpt2_classifier.train import train_classifier_simple

device = get_device()
model.to(device)

start_time = time.time()

torch.manual_seed(123)
# optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
optim_groups = get_adam_param_groups(model, weight_decay=0.1)
optimizer = torch.optim.AdamW(optim_groups, lr=5e-5)

num_epochs = 5
(
    train_losses,
    val_losses,
    train_accs,
    val_accs,
    train_precisions,
    val_precisions,
    train_roc_aucs,
    val_roc_aucs,
    train_pr_aucs,
    val_pr_aucs,
    examples_seen
    ) = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=10,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

### **Visualize Finetuning Metrics**

In [ ]:
from gpt2_classifier.utils import plot_results

fig_path = plot_results(
    num_epochs,
    train_losses,
    val_losses,
    train_accs,
    val_accs,
    train_precisions,
    val_precisions,
    train_roc_aucs,
    val_roc_aucs,
    train_pr_aucs,
    val_pr_aucs,
    examples_seen
)

print(fig_path)

---

## **Compute and Visualize Fine-tuned Model's Metrics**

In [ ]:
from gpt2_classifier.evaluate import calc_classification_metrics_loader


train_metrics = calc_classification_metrics_loader(train_loader, model, device, num_batches=20)
val_metrics = calc_classification_metrics_loader(val_loader, model, device, num_batches=20)
test_metrics = calc_classification_metrics_loader(test_loader, model, device, num_batches=20)

print("Training metrics:")
print(
    f"Accuracy: {train_metrics.accuracy:.3f} | Precision: {train_metrics.precision:.3f} | ROC-AUC: {train_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {train_metrics.pr_auc:.3f}\n"
)

print("Validation metrics:")
print(
    f"Accuracy: {val_metrics.accuracy:.3f} | Precision: {val_metrics.precision:.3f} | ROC-AUC: {val_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {val_metrics.pr_auc:.3f}\n"
)

print("Test metrics:")
print(
    f"Accuracy: {test_metrics.accuracy:.3f} | Precision: {test_metrics.precision:.3f} | ROC-AUC: {test_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {test_metrics.pr_auc:.3f}\n"
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# Combine all metrics for Train, Validation, and Test sets in one dataframe
metrics_df = pd.DataFrame(
    {
        'Train': dict(train_metrics),
        'Validation': dict(val_metrics),
        'Test': dict(test_metrics),
    },
    index=pd.Index(['accuracy', 'precision', 'roc_auc', 'pr_auc'], name='Metric')
).reset_index()

# Pivot dataframe suitable for plotting
metrics_df_long = pd.melt(
    metrics_df, id_vars=['Metric'], value_vars=['Train', 'Validation', 'Test'], var_name='Set', value_name='Value')

metrics_df_long
fig, ax = plt.subplots(figsize=(5,4))

sns.barplot(ax=ax, data=metrics_df_long, x='Metric', y='Value', hue='Set')
ax.set_title("Classification Metrics")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
ax.legend(title="Set", loc="lower right")
plt.show()
fig.savefig(paths.MODELS_DIR / "classification_metrics.pdf", dpi=300, bbox_inches='tight');

---

## **Inference**

In [ ]:
from gpt2_classifier.inference import classify_text

text_2 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_text(
    text_2, model, tokenizer, device, max_length=120
))

In [ ]:
from gpt2_classifier.inference import classify_text

text_3 = (
    "Hey Smith, how are you doing? Let's catch up sometime!"
)

print(classify_text(
    text_3, model, tokenizer, device, max_length=120
))